# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook demonstrates loading and exploring the FAIR^2 dataset using the `mlcroissant` library and Croissant schema. The workflow covers data loading, reviewing Croissant entities via `@id`, extraction, processing, and visualization.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json`

In [ ]:
# Ensure the mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print('Dataset Title:', metadata.name)
print('Dataset Description:', metadata.description)
print('Dataset Published:', metadata.datePublished)

## 2. Data Overview
Review available record sets, fields, and their `@id` values.

The Croissant schema encodes structured tabulations, which are typically mapped to record sets. Let's enumerate all record sets, fields, and columns (all are referenced by their `@id`).

In [ ]:
# List all record sets (by @id), fields and columns
record_sets = list(dataset.record_sets)
print('Available record sets:')
for rs in record_sets:
    print(f"- Record set @id: {rs['@id']}, name: {rs.get('name', 'N/A')}")

# For demonstration, extract all fields and columns for each record set
for rs in record_sets:
    print(f"\nRecord set: {rs['@id']}")
    fields = rs.get('fields', [])
    for f in fields:
        print(f"  Field @id: {f['@id']}, name: {f.get('name', 'N/A')}, dataType: {f.get('dataType', 'N/A')}")
        columns = f.get('columns', [])
        for col in columns:
            print(f"    Column @id: {col['@id']}, name: {col.get('name', 'N/A')}")

## 3. Data Extraction
Load data from one or more record sets into Pandas DataFrames for analysis. All entities are referenced by their `@id` exactly as they appear in the Croissant schema.

In [ ]:
# Example: Extracting data from all available record sets

# Collect all record set @ids
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded DataFrame for Record Set @id: {rs_id} with {len(df)} rows.")
        print('Columns:', df.columns.tolist())
        print(df.head())
    except Exception as e:
        print(f"Could not load records for {rs_id}: {e}")

## 4. Exploratory Data Analysis (EDA)
Perform basic data processing for selected fields and columns. Use entity `@id`s for referencing specific fields.

Typical steps include outlier removal, normalization, or grouping.

In [ ]:
# Choose one record set (e.g. the first one)
if record_set_ids:
    selected_record_set_id = record_set_ids[0]
    df = dataframes[selected_record_set_id]
    print(f"EDA on Record Set @id: {selected_record_set_id}")

    # Find a numeric field @id (e.g. look for 'age' or 'hormone_level', depending on schema)
    numeric_columns = [col for col in df.columns if df[col].dtype in [np.float64, np.int64]]
    if numeric_columns:
        numeric_field_id = numeric_columns[0]
        print(f"Selected numeric field @id: {numeric_field_id}")

        threshold = df[numeric_field_id].mean()
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try to group by a categorical field (by @id)
        group_field_id = None
        for col in df.columns:
            if df[col].dtype == object and df[col].nunique() < 5:
                group_field_id = col
                break
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped data by {group_field_id} (@id):")
            print(grouped_df.head())
    else:
        print("No numeric fields found in the selected DataFrame.")

## 5. Visualization
Visualize distributions or relationships between selected fields and columns.

In [ ]:
# Example: histogram of selected numeric field
if record_set_ids and numeric_columns:
    plt.figure(figsize=(6,4))
    df[numeric_field_id].hist(bins=10)
    plt.title(f"Distribution of {numeric_field_id} (@id)")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    # If grouped_df is available, plot a bar chart
    if 'grouped_df' in locals():
        grouped_df[numeric_field_id].plot(kind='bar')
        plt.title(f"Mean {numeric_field_id} by {group_field_id} (@id)")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.show()

## 6. Conclusion
This notebook illustrated stepwise exploration of a clinical and genotype–phenotype dataset using Croissant schema and the `mlcroissant` library.

Key Steps:
- Croissant entities are referenced by their `@id`, enabling precise extraction and manipulation.
- Data loading and inspection with `mlcroissant` provides both structural and record-level views.
- Exploratory analysis and visualizations are possible even for small case-based datasets, aiding reproducibility and further research.

For further exploration, examine additional record sets, fields or visualizations by their `@id`, and apply more domain-specific analyses.